## Selecting the Model & Evaluating the Base Model

Base Model: 8 Billion Params, quantized down to 4 bits compared with GPT-5 at TRILLIONS of params!

In [20]:
# !pip install --upgrade bitsandbytes

In [21]:
!wget https://raw.githubusercontent.com/ed-donner/llm_engineering/main/week7/util.py -O util.py

--2026-10-18 15:06:04--  https://raw.githubusercontent.com/ed-donner/llm_engineering/main/week7/util.py
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.109.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 6917 (6.8K) [text/plain]
Saving to: ‘util.py’

util.py             100%[===================>]   6.75K  --.-KB/s    in 0s      

2026-10-18 15:06:05 (75.8 MB/s) - ‘util.py’ saved [6917/6917]



In [22]:
# libraries
import os
import re
import math
from tqdm import tqdm
from huggingface_hub import login
import torch
import transformers
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig, TrainingArguments, set_seed
from peft import LoraConfig, PeftModel
from datasets import load_dataset, Dataset, DatasetDict
from datetime import datetime
import matplotlib.pyplot as plt
from util import evaluate

In [23]:
# setting constants and environments
BASE_MODEL = 'meta-llama/Llama-3.2-3B'
PROJECT_NAME = 'price'

LITE_MODE = False

DATA_USER = 'irene12'
DATASET_NAME = f"{DATA_USER}/items_prompts_lite" if LITE_MODE else f"{DATA_USER}/items_prompts_full"

QUANT_4_BIT = True

In [ ]:
hf_token = 'HF_TOKEN'

login(hf_token)

### Loading the Data

In [25]:
dataset = load_dataset(DATASET_NAME)
train = dataset['train']
val = dataset['val']
test = dataset['test']

In [26]:
test[0]

{'prompt': 'What does this cost to the nearest dollar?\n\nTitle: Excess V2 Distortion/Modulation Pedal  \nCategory: Music Pedals  \nBrand: Old Blood Noise  \nDescription: A versatile pedal offering distortion and three modulation modes—delay, chorus, and harmonized fifths—with full control over signal routing and expression.  \nDetails: Features include separate gain, tone, and volume controls; time, depth, and volume per modulation; order switching, soft‑touch bypass, and expression jack for dynamic control.\n\nPrice is $',
 'completion': '219.0'}

### Preparing the Base Llama Model for Evaluation

In [27]:
if QUANT_4_BIT:
    quant_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_use_double_quant=True,
        bnb_4bit_compute_dtype=torch.bfloat16,
        bnb_4bit_quant_type='nf4'
    )
else:
    quant_config = BitsAndBytesConfig(
        load_in_8bit=True,
        bnb_4bit_compute_dtype=torch.bfloat16
    )

In [28]:
# loading the tokenizer and the model
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = 'right'

base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    quantization_config=quant_config,
    device_map='auto'
)

base_model.generation_config.pad_token_id = tokenizer.pad_token_id

print(f"Memory footprint: {base_model.get_memory_footprint() / 1e9:.1f} GB")

Loading weights:   0%|          | 0/254 [00:00<?, ?it/s]

Memory footprint: 2.2 GB


In [29]:
def model_predict(item):
    inputs = tokenizer(item['prompt'], return_tensors="pt").to("cuda")

    with torch.no_grad():
        output_ids = base_model.generate(**inputs, max_new_tokens=8)

    prompt_len = inputs['input_ids'].shape[1]
    generated_ids = output_ids[0, prompt_len:]

    return tokenizer.decode(generated_ids)


In [30]:
print(test[0])

{'prompt': 'What does this cost to the nearest dollar?\n\nTitle: Excess V2 Distortion/Modulation Pedal  \nCategory: Music Pedals  \nBrand: Old Blood Noise  \nDescription: A versatile pedal offering distortion and three modulation modes—delay, chorus, and harmonized fifths—with full control over signal routing and expression.  \nDetails: Features include separate gain, tone, and volume controls; time, depth, and volume per modulation; order switching, soft‑touch bypass, and expression jack for dynamic control.\n\nPrice is $', 'completion': '219.0'}


In [31]:
model_predict(test[0])

'169 USD, and it is available now'

In [32]:
from importlib import reload

### Evaluation

In [33]:
evaluate(model_predict, test)